In [1]:
import e57
import open3d as o3d
import numpy as np

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
pc = e57.read_points(r"C:\Users\mvm\open3d_vision\data\67.e57")
print(pc.points)

[[11.520998   -0.59979987 -2.40382075]
 [11.30600262 -0.29670337 -2.50264382]
 [11.42894936 -0.47084644 -2.50669765]
 ...
 [11.62114811  0.23616765 -1.3764863 ]
 [11.71813107  0.31359017 -1.41600394]
 [11.69162464  0.28622872 -1.29827034]]


In [3]:
# Convertir et afficher le nuage de points dans Open3D
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pc.points)
if hasattr(pc, "color") and pc.color is not None:
    pcd.colors = o3d.utility.Vector3dVector(pc.color)

In [4]:
def revert_normals(pcd):
    pcd.normals = -pcd.normals
    return pcd

In [5]:
# Géométrisation Poisson (performances optimisées)
import numpy as np

def _ram_mb():
    """RAM utilisée par le processus, en Mo (RSS). Retourne None si psutil absent."""
    try:
        import psutil
        return psutil.Process().memory_info().rss / (1024 * 1024)
    except ImportError:
        return None

def _log_ram(etape):
    m = _ram_mb()
    print(f"[RAM] {etape}: {m:.1f} Mo" if m is not None else f"[RAM] {etape}: (installer psutil pour afficher)")

_log_ram("Début (nuage pcd)")

# Sous-échantillonnage si > 500k points pour accélérer
VOXEL_SIZE = 0.01  # ajuster selon la densité du scan
if len(pcd.points) > 500_000:
    pcd_work = pcd.voxel_down_sample(VOXEL_SIZE)
    print(f"Sous-échantillonné: {len(pcd.points)} -> {len(pcd_work.points)} points")
else:
    pcd_work = pcd
_log_ram("Après sous-échantillonnage")

# Normales (KNN léger pour la vitesse)
pcd_work.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamKNN())
pcd_work.orient_normals_consistent_tangent_plane(100)
t = np.asarray(pcd_work.normals)
pcd_work.normals = o3d.utility.Vector3dVector(-t)
_log_ram("Après estimation normales")

# Reconstruction Poisson (depth=9 : compromis qualité/vitesse)
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd_work, depth=9)
_log_ram("Après Poisson (mesh + densities)")

# Retrait des vertices en zones de faible densité (artefacts)
densities = np.asarray(densities)
q = np.quantile(densities, 0.01)
vertices_to_remove = densities < q
mesh.remove_vertices_by_mask(vertices_to_remove)
_log_ram("Après retrait vertices faible densité")

# Transfert des couleurs du nuage vers le mesh (point le plus proche)
if pcd_work.has_colors():
    from scipy.spatial import cKDTree
    tree = cKDTree(np.asarray(pcd_work.points))
    _, idx = tree.query(np.asarray(mesh.vertices), k=1, workers=-1)
    mesh.vertex_colors = o3d.utility.Vector3dVector(np.asarray(pcd_work.colors)[idx])
_log_ram("Après transfert couleurs")

print(f"Mesh: {len(mesh.vertices)} vertices, {len(mesh.triangles)} triangles")
o3d.visualization.draw_geometries([mesh], window_name="Surface Poisson colorée")

[RAM] Début (nuage pcd): 1102.0 Mo
Sous-échantillonné: 11532410 -> 1005833 points
[RAM] Après sous-échantillonnage: 1147.1 Mo
[RAM] Après estimation normales: 1176.0 Mo
[RAM] Après Poisson (mesh + densities): 1221.7 Mo
[RAM] Après retrait vertices faible densité: 1222.4 Mo
[RAM] Après transfert couleurs: 1261.2 Mo
Mesh: 357244 vertices, 713109 triangles


In [6]:
o3d.io.write_triangle_mesh("pcd_work.ply", mesh)

True

In [12]:
o3d.visualization.draw_geometries([mesh], window_name="Surface Poisson colorée")

In [8]:
test = o3d.io.read_triangle_mesh("pcd_work.ply")

In [9]:
o3d.visualization.draw_geometries([test])

In [10]:
def set_vertex_color(mesh, vertex_index, color_rgb):
    """Définit la couleur du vertice `vertex_index` comme `color_rgb` sur le mesh open3d."""
    import numpy as np
    verts = np.asarray(mesh.vertices)
    n = len(verts)
    # S'assurer que le mesh a une liste de couleurs pour chaque sommet
    vertex_colors = np.asarray(mesh.vertex_colors)
    if vertex_colors.shape[0] != n:
        # Initialise une couleur par sommet (par défaut : blanc)
        mesh.vertex_colors = o3d.utility.Vector3dVector(np.ones((n, 3)))
        vertex_colors = np.asarray(mesh.vertex_colors)
    # Applique la couleur voulue
    vertex_colors[vertex_index] = color_rgb

for i in range(10000):
    set_vertex_color(test, i, [1.0, 0.0, 0.0])
o3d.visualization.draw_geometries([test])

In [11]:
# données manquantes: position de la caméra, orientation de la caméra ?
